# Project: Customer Support Chatbot for a Retail Store

### Learning objectives:
- Build a RAG system using the below building blocks:
  - Ingest docs and split docs into chunks
  - Embed chunks and store in vector DB
  - Build the retriever tool and RAG pipeline to combine everything and call LLM
- Build a SQL database agent using the below building blocks:
  - Use python's postgreSQL MCP server to call postgreSQL DB
  - Call MCP server to answer user's questions on sales orders
- Using langchain tool calling, have a LLM interpret user's question and
  - Call the RAG system to answer questions related to store policies
  - Call the SQL agent to answer questions related to orders

### High Level Steps:
1. Prepare virtual environment
2. Data preparation for RAG system
   1. Load source docs
   2. Chunk docs
3. Build Retriever
   1. Generate embeddings
   2. Create vector DB
4. Tool 1: RAG engine
   1. Connect system prompt, retriever and LLM together
5. Tool 2: SQL agent
   1. Fetch MCP parameters
   2. Load tools
   3. Invoke agent and LLM
6. Chainlit UI






>

## 1. Prepare environment

#### Step 1: Create your environment and install dependencies

Install uv (skip if already installed)

> curl -LsSf https://astral.sh/uv/install.sh | sh

Create venv and install dependencies

> uv venv .venv-cust-chat-uv && source .venv-cust-chat-uv/bin/activate

> uv pip install -r requirements.txt


#### Step 2: Register this environment as a Jupyter kernel

```python
python -m ipykernel install --user --name="cust-chat-uv" --display-name "cust-chat-uv"
```

In [ ]:
# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Generate text embeddings using OpenAI or Hugging Face models
from langchain_openai import OpenAIEmbeddings

# Create prompts for the RAG system
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

from openai import OpenAI
from ddgs import DDGS
from langchain_core.tools import tool
from langchain.agents import create_agent

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv(override=True)
#os.environ["NODE_TLS_REJECT_UNAUTHORIZED"] = "0"

print("✅ Libraries imported! You're good to go!")

## 2: Data preparation for RAG system

In [ ]:
########################################################################################################################
# LOAD DOCUMENT
########################################################################################################################

raw_filename = 'retail-store-help-desk-data.md'
loader = TextLoader(raw_filename, encoding="utf-8")
docs = loader.load()
text = docs[0].page_content
print(len(text))


In [ ]:
########################################################################################################################
# SPLIT DOCUMENT INTO CHUNKS
########################################################################################################################

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("###", "id")],
    strip_headers=True)

chunked_docs = splitter.split_text(text)
print(len(chunked_docs), "Q/A chunks")
#print(chunked_docs[0])
#print(chunked_docs[0].page_content)

## 3. Build Retriever

In [ ]:
########################################################################################################################
# TURN EACH CHUNK INTO AN EMBEDDING VECTOR & STORE IN VECTOR DB
########################################################################################################################

# create the embeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# # create vector database
# vectorstore = Chroma.from_documents(documents=chunked_docs,
#                                     embedding=embeddings,
#                                     collection_metadata={"hnsw:space": "cosine"},
#                                     persist_directory="retail_vector_db_chroma",
#                                     collection_name="retail_help_qa")

# code to load DB once saved

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory="retail_vector_db_chroma",
                     collection_name="retail_help_qa",
                     embedding_function=embeddings)


In [ ]:
########################################################################################################################
# Create a retriever object with a suitable k value (e.g., 8)
########################################################################################################################

#retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

retriever = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"k": 6,  "score_threshold": 0.25})

## 4. Tool 1: RAG engine

In [ ]:
########################################################################################################################
# PREPARE RAG SYSTEM PROMPT
########################################################################################################################

# bring in the system instructions
with open("rag-system-prompt.txt", "r", encoding="utf-8") as f:
    rag_system_text = f.read()

prompt = ChatPromptTemplate.from_template(rag_system_text)

In [ ]:
########################################################################################################################
# Define tool for RAG generation
########################################################################################################################

llm = ChatOpenAI(model="gpt-4.1", temperature=0)

def format_docs(docs):
    contents = [d.page_content for d in docs]
    return "\n\n".join(contents)

@tool
def rag_agent(question: str):
    """Use this tool for questions about store policies, returns, or general FAQs."""
    
    # RAG answer chain: {input} -> retrieve -> format -> prompt -> model -> string
    # In the context of a RAG pipeline, RunnableLambda(format_docs) is a bridge that transforms the raw output of a retriever (list of document objects) into a format the LLM can understand
    # RunnableLambda does Type Conversion + Chain Integration + allows Observability in langsmith - this formatting step will show up as its own named block, allowing you to see exactly what "context" was sent to the prompt after retrieval
    rag_answer_chain = (
        {
            "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
            "question": itemgetter("input"),    
        }
        | prompt
        | llm
    )

    response = rag_answer_chain.invoke({"input": question})
    return response.content


In [ ]:
question = "What is your refund policy?"
# question = ("What products do you sell?")
answer = rag_agent(question)
print(answer)

## Tool 2: SQL Agent

In [ ]:
########################################################################################################################
# Fetch MCP parameters
########################################################################################################################

fs_params = StdioServerParameters(
    command="npx", 
    args=["-y", "@modelcontextprotocol/server-postgres", f"postgresql://postgres:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DBNAME')}?sslmode=no-verify"]
)


In [ ]:
########################################################################################################################
# PREPARE SQL SYSTEM PROMPT
########################################################################################################################

# bring in the system instructions
with open("sql-system-prompt.txt", "r", encoding="utf-8") as f:
    sql_system_text = f.read()

In [ ]:
########################################################################################################################
# Load Tools and call Parent Agent
########################################################################################################################

async with stdio_client(fs_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        mcp_tools = await load_mcp_tools(session)

        sql_agent = create_agent(
            llm, 
            tools=mcp_tools, 
            system_prompt=sql_system_text
        )

        @tool
        async def sales_db_tool(query: str):
            """Use this for order status, shipping status or sales data."""

            try:

                response = await sql_agent.ainvoke({"messages": [("user", query)]})
                return response["messages"][-1].content
            except Exception as e:
                return f"Error accessing database: {str(e)}"

        # Initialize the parent Agent
        parent_agent = create_agent(llm, 
                                    tools=[sales_db_tool, rag_agent],
                                    system_prompt="You are a retail assistant. Route queries to the order expert or policy expert as needed.")

        
        # Example 1: Routes to PostgreSQL MCP
        #res1 = await parent_agent.ainvoke({"messages": [("user", "What is the total sales value for Order_ID = 'CA-2019-100678'?")]})
        #res1 = await parent_agent.ainvoke({"messages": [("user", "How many orders were placed for 06-01-2019 ?")]})
        #res1 = await parent_agent.ainvoke({"messages": [("user", "What is the shipping status for Order_ID = 'CA-2019-103800'?")]})
        #print(f"Order Query: {res1['messages'][-1].content}")

        # Example 2: Routes to RAG Tool
        res2 = await parent_agent.ainvoke({"messages": [("user", "How do I return an item?")]})
        #res2 = await parent_agent.ainvoke({"messages": [("user", "What is your refund policy?")]})
        print(f"Customer Support Query: {res2['messages'][-1].content}")

Customer Support Query: We offer men's, women's, and kids' clothing, casual footwear, pantry and household goods, catering trays, and eligible food items. Please note that selection and availability may vary by location and whether you shop in-store or through our app. If you have a specific product in mind, let me know and I can provide more details!


## Chainlit UI Solution

In [17]:
%%writefile chainlit_chatbot_app.py
# ---------------------------------------------------------
# Chainlit Retail Store Assistant

import chainlit as cl
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp import ClientSession, StdioServerParameters
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_core.messages import AIMessage, ToolMessage
from langchain.tools import tool
from dotenv import load_dotenv
from langchain_core.runnables import RunnableLambda
from operator import itemgetter
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from langgraph_supervisor import create_supervisor
import os
import asyncio

load_dotenv(override=True)

########################################################################################################################
# PREPARE RAG SYSTEM PROMPT
########################################################################################################################

# bring in the system instructions
with open("rag-system-prompt.txt", "r", encoding="utf-8") as f:
    rag_system_text = f.read()


########################################################################################################################
# PREPARE SQL SYSTEM PROMPT
########################################################################################################################

# bring in the system instructions
with open("sql-system-prompt.txt", "r", encoding="utf-8") as f:
    sql_system_text = f.read()


########################################################################################################################
# Load the vector DB with embeddings
########################################################################################################################

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma(persist_directory="retail_vector_db_chroma",
                     collection_name="retail_help_qa",
                     embedding_function=embeddings)

########################################################################################################################
# Create a retriever object with a suitable k value (e.g., 8)
########################################################################################################################

#retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

retriever = vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs={"k": 6,  "score_threshold": 0.25})



########################################################################################################################
# Initialize LLM model
########################################################################################################################
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

########################################################################################################################
# Define tool for RAG generation
########################################################################################################################

prompt = ChatPromptTemplate.from_template(rag_system_text)

def format_docs(docs):
    contents = [d.page_content for d in docs]
    return "\n\n".join(contents)

@tool
async def rag_agent(question: str):
    """Use this tool for questions about store policies, returns, or general FAQs."""
    
    # RAG answer chain: {input} -> retrieve -> format -> prompt -> model -> string
    # In the context of a RAG pipeline, RunnableLambda(format_docs) is a bridge that transforms the raw output of a retriever (list of document objects) into a format the LLM can understand
    # RunnableLambda does Type Conversion + Chain Integration + allows Observability in langsmith - this formatting step will show up as its own named block, allowing you to see exactly what "context" was sent to the prompt after retrieval
    rag_answer_chain = (
        {
            "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
            "question": itemgetter("input"),    
        }
        | prompt
        | llm
    )
    response = await rag_answer_chain.ainvoke({"input": question})
    return response.content

########################################################################################################################
# Code for chainlit UI
########################################################################################################################

@cl.on_chat_start
async def start():
    mcp_tools = []

    # Setup MCP Connection
    # Configure the servers in a dictionary format
    server_config = {
        "postgres": {
            "command": "npx",
            "args": [
                "-y", 
                "@modelcontextprotocol/server-postgres", 
                f"postgresql://postgres:{os.getenv('POSTGRES_PASSWORD')}@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DBNAME')}?sslmode=no-verify"
            ],
            "transport": "stdio"
        }
    }

    try:
        # Initialize the MultiServer client
        client = MultiServerMCPClient(server_config)
        cl.user_session.set("mcp_client", client)
        
        mcp_tools = await asyncio.wait_for(client.get_tools(), timeout=15.0)
        print(f"Successfully loaded {len(mcp_tools)} tools from MCP")

    except Exception as e:
        await cl.Message(content=f"⚠️ Connection failed: {str(e)}").send()
        mcp_tools = []

    sql_agent = create_agent(
            llm, 
            tools=mcp_tools, 
            system_prompt=sql_system_text
        )

    @tool
    async def sales_db_tool(query: str):
        """Use this for order status, shipping status or sales data."""

        try:

            response = await sql_agent.ainvoke({"messages": [("user", query)]})
            return response["messages"][-1].content
        except Exception as e:
            return f"Error accessing database: {str(e)}"
    
    # Initialize the parent Agent
    parent_agent = create_agent(ChatOpenAI(model="gpt-4.1", temperature=0), 
                                tools=[sales_db_tool, rag_agent],
                                system_prompt="You are a retail assistant. Route queries to the order expert or policy expert as needed."
                                )
    cl.user_session.set("parent_agent", parent_agent)

    await cl.Message(content="Hello! I'm your retail assistant. How can I help with your order or our policies today?").send()

# --- Message Handling ---
@cl.on_message
async def handle_message(message: cl.Message):
    """Handle user messages and stream agent responses."""
    print("Inside on message func")
    final_answer = ""
    agent = cl.user_session.get("parent_agent")
    
    res = await agent.ainvoke({"messages": [("user", message.content)]})
    
    await cl.Message(content=res["messages"][-1].content).send()

@cl.on_chat_end
async def end():
    # Cleanup MCP connection
    transport = cl.user_session.get("mcp_transport")
    if transport:
        await transport.__aexit__(None, None, None)

Overwriting chainlit_chatbot_app.py
